# Análise da segurança pública de Joinville/SC

SENAI/LAB365 — Visualização de Dados e BI. Execute as células em ordem, com o kernel da `.venv`.
Cada célula é uma etapa; as explicações estão nos comentários `#` e os resultados saem de `print()` e `display()`.

- **Sinesp/MJSP** (etapas 2 a 11): base aprovada, 911 observações, 12 séries, janeiro/2020 a setembro/2026 (PRF até agosto).
- **SSP/SC** (etapas 12 a 15): investigação **separada** de violência doméstica, em validação. Não foi identificada uma definição operacional de "Fato Comunicado" nos documentos analisados (isso não significa que a definição não exista oficialmente); por isso a unidade não é lida como vítima nem como ocorrência única.
- As duas fontes nunca são somadas nem unidas. O [README](../README.md) reúne fontes, regras e pendências.

In [ ]:
# ETAPA 1 — Bibliotecas e pastas
import hashlib
import sys
from pathlib import Path

import pandas as pd

# O notebook roda dentro da pasta notebooks/; a pasta-raiz entra no caminho do Python para importar os módulos do projeto.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from dashboard import calculos, graficos            # cálculos e gráficos já validados (os mesmos do painel)
from dashboard import dados_publicos as dp          # leitura e conferência da base aprovada
from src.pipeline import atualizacao_dados as au    # leitura do XLSX oficial

pasta_xlsx = RAIZ / "dados" / "brutos" / "sinesp"   # XLSX oficiais (originais, fora do Git)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
%matplotlib inline

print(f"Pandas {pd.__version__} | pasta do projeto: {RAIZ}")

In [ ]:
# ETAPA 2 — Leitura da base Sinesp aprovada
# É o CSV publicado no painel (recorte de Joinville/SC). dp.carregar_sinesp() lê com pd.read_csv e confere colunas, tipos e regras.
dados_sinesp = dp.carregar_sinesp()
manifesto = dp.carregar_manifesto()

print(f"Base aprovada: {dados_sinesp.shape[0]} observações × {dados_sinesp.shape[1]} colunas")
print(f"Versão dos dados: {manifesto['versao']}")
# O manifesto guarda o SHA-256 do CSV; sha256_normalizado calcula o mesmo valor para conferirmos que o arquivo não mudou.
hash_csv = dp.sha256_normalizado((dp.DIRETORIO_DADOS / dp.ARQ_SINESP).read_bytes())
hash_manifesto = next(f["sha256"] for f in manifesto["arquivos"] if f["arquivo"] == dp.ARQ_SINESP)
print(f"SHA-256 do CSV confere com o manifesto: {hash_csv == hash_manifesto}")
display(dados_sinesp.head())

In [ ]:
# ETAPA 3 — XLSX oficial de 2026 (opcional): leitura e inspeção bruta
# O ETL (scripts/etl_joinville.py) lê os sete XLSX. Aqui lemos só o de 2026 para mostrar como a fonte original chega.
# Só usamos o arquivo cujo SHA-256 é o registrado no manifesto da base aprovada.
hash_2026 = manifesto["origem_dos_dados"]["sinesp_xlsx_sha256"]["2026"]
dados_brutos_2026 = None
if pasta_xlsx.is_dir():
    for caminho in sorted(pasta_xlsx.rglob("bancovde-2026.xlsx")):
        if hashlib.sha256(caminho.read_bytes()).hexdigest() == hash_2026:
            # ler_sinesp_xlsx devolve só as linhas de SC/JOINVILLE, com as 14 colunas originais.
            dados_brutos_2026 = au.ler_sinesp_xlsx(caminho)
            break

if dados_brutos_2026 is None:
    # Sem o arquivo oficial não há inspeção bruta: a análise segue com a base aprovada (dado derivado, não bruto).
    print("MODO DEMONSTRAÇÃO: XLSX oficial de 2026 não encontrado; a análise segue com a base aprovada.")
else:
    print(f"XLSX 2026 lido: {len(dados_brutos_2026)} linhas de Joinville")
    # Na fonte, a data é um número (serial do Excel); total_vitima traz as vítimas e total traz os mandados.
    dados_brutos_2026.info()
    display(dados_brutos_2026.head())

In [ ]:
# ETAPA 4 — Tratamento: tipos e unidades
# No ETL cada linha original virou uma observação com ano, mes, evento, abrangencia, unidade e valor.
# Vítimas e mandados são unidades diferentes e nunca são somadas.
print(dados_sinesp.dtypes)
print()
# value_counts conta quantas observações há de cada unidade.
print(dados_sinesp["unidade"].value_counts())
# Int64 (I maiúsculo) aceita valor ausente (<NA>) sem trocá-lo por número.
print(f"\nTipo da coluna valor: {dados_sinesp['valor'].dtype}")

In [ ]:
# ETAPA 5 — Verificação de duplicidades
# Cada série (evento × abrangência) deve ter UMA observação por mês. Nenhuma linha é apagada.
# Se houvesse repetição, ela seria investigada (o Sinesp só admite parcelas somadas em Tentativa de homicídio e Estupro).
chave_observacao = ["evento", "abrangencia", "ano", "mes"]
repetidas = dados_sinesp.duplicated(subset=chave_observacao, keep=False)
print(f"Observações com chave repetida: {int(repetidas.sum())}")
print(f"Linhas totalmente idênticas: {int(dados_sinesp.duplicated().sum())}")

In [ ]:
# ETAPA 6 — Valores ausentes e zeros
# Zero publicado = a fonte informou 0 no mês (observação válida). Ausente (<NA>) = a fonte não informou; nunca vira zero.
print(f"Valores ausentes: {int(dados_sinesp['valor'].isna().sum())}")
print(f"Zeros publicados: {int(dados_sinesp['valor'].eq(0).sum())}")

# Zeros por série: eventos raros (latrocínio, feminicídio) têm muitos meses com zero.
dados_com_zero = dados_sinesp.assign(zero=dados_sinesp["valor"].eq(0))
zeros_por_serie = dados_com_zero.groupby(["evento", "abrangencia"])["zero"].sum()
display(zeros_por_serie.sort_values(ascending=False).to_frame("meses com zero"))

In [ ]:
# ETAPA 7 — Recorte de Joinville e cobertura das séries
# Toda a base é de Joinville/SC. O catálogo mostra o primeiro e o último mês de cada série.
# A PRF termina em agosto/2026 e os mandados da Polícia Federal começam em janeiro/2025.
catalogo = dp.carregar_catalogo_sinesp()
print(f"Séries no catálogo: {len(catalogo)}")
display(catalogo[["evento", "abrangencia", "unidade", "inicio", "fim", "meses_observados", "meses_valor_zero"]])

In [ ]:
# ETAPA 8 — Estatísticas descritivas (homicídio doloso, Estadual)
# Descrevem os valores MENSAIS: soma, média, mediana, mínimo, máximo e desvio-padrão (ddof=0: descreve os meses observados).
INDICADOR = "Homicídio doloso"
ABRANGENCIA = "Estadual"
ANOS = list(range(2020, 2027))

serie = calculos.serie_sinesp(dados_sinesp, INDICADOR, ABRANGENCIA)
resumo = calculos.resumo_serie_sinesp(serie)
display(pd.Series(resumo).to_frame("valor"))

# Quartis: Q1 e Q3 são os valores abaixo dos quais ficam 25% e 75% dos meses; IQR = Q3 − Q1.
quartis = calculos.quartis_iqr(serie["valor"])
print(f"Q1 = {quartis['q1']} | mediana = {quartis['mediana']} | Q3 = {quartis['q3']} | IQR = {quartis['iqr']}")
print(f"Média mensal {resumo['media_mensal']:.2f} e mediana {resumo['mediana_mensal']:.0f}: poucos meses muito altos puxam a média.")

In [ ]:
# ETAPA 9 — Totais anuais e períodos equivalentes
# Só os anos completos (12 meses) recebem total anual; 2026 é parcial.
tabela_anual = calculos.tabela_anual_sinesp(serie)
display(tabela_anual[["ano", "cobertura", "meses_presentes", "soma"]])

# Para comparar 2026 com os outros anos, usamos o mesmo período: janeiro até o último mês publicado (setembro).
mes_final = calculos.mes_final_da_serie(serie)
periodos = calculos.comparacao_periodo_sinesp(serie, mes_final)
display(periodos[["ano", "periodo", "soma"]])

# Interpretação calculada direto das tabelas (descreve as contagens; não explica causas).
total_por_ano = tabela_anual.set_index("ano")["soma"]
total_periodo = periodos.set_index("ano")["soma"]
print(f"Anos completos: {int(total_por_ano[2020])} vítimas em 2020 e {int(total_por_ano[2025])} em 2025 "
      f"(com oscilações: {int(total_por_ano[2022])} em 2022 e {int(total_por_ano[2023])} em 2023).")
print(f"Janeiro–setembro: {int(total_periodo[2025])} vítimas em 2025 e {int(total_periodo[2026])} em 2026 "
      f"(diferença de {int(total_periodo[2026] - total_periodo[2025])}).")

In [ ]:
# ETAPA 10 — Outliers pelo método IQR (limites de Tukey)
# Limite inferior = Q1 − 1,5·IQR; limite superior = Q3 + 1,5·IQR. Meses fora deles são atípicos ESTATÍSTICOS, não erros comprovados.
diagnostico = calculos.diagnostico_outliers_iqr(serie)
print(f"{diagnostico['rotulo']} — {diagnostico['mensagem']}")
print(f"Limites: {diagnostico['limite_inferior']} a {diagnostico['limite_superior']} | meses fora: {diagnostico['n_fora']}")
display(diagnostico["fora"])

# O método não serve para todas as séries (IQR igual a zero, maioria de zeros, contagens muito baixas).
diagnosticos = calculos.diagnostico_outliers_todas_as_series(dados_sinesp)
display(diagnosticos[["evento", "abrangencia", "meses", "zeros", "classificacao"]])
informativas = int(diagnosticos["classificacao"].eq("método informativo").sum())
print(f"Método informativo em {informativas} das {len(diagnosticos)} séries; nas demais nenhum mês é apontado como atípico.")

In [ ]:
# ETAPA 11 — Gráficos do Sinesp (os mesmos do painel)
# Evolução anual (barras).
display(graficos.figura_serie_anual_sinesp(dados_sinesp, INDICADOR, ABRANGENCIA, ANOS))
print(f"Anual: {int(total_por_ano[2020])} vítimas em 2020 e {int(total_por_ano[2025])} em 2025; 2026 aparece como parcial.")

# Evolução mensal de 2025 e 2026.
display(graficos.figura_mensal_sinesp(dados_sinesp, INDICADOR, ABRANGENCIA, [2025, 2026]))
print("Mensal: contagens pequenas oscilam bastante; a comparação correta é entre períodos de mesma duração.")

# Perfil por mês do ano (mediana), usando só os anos completos.
display(graficos.figura_sazonalidade_sinesp(dados_sinesp, INDICADOR, ABRANGENCIA, ANOS))
anos_usados = calculos.perfil_sazonal(serie).attrs["anos_usados"]
print(f"Perfil mensal: usa {len(anos_usados)} anos completos ({anos_usados[0]}–{anos_usados[-1]}); descreve esses anos e não prova sazonalidade.")

# Série mensal com os limites de Tukey.
display(graficos.figura_outliers_iqr_sinesp(dados_sinesp, INDICADOR, ABRANGENCIA, ANOS))
fora = diagnostico["fora"]
meses_fora = ", ".join(f"{int(l.mes):02d}/{int(l.ano)} ({int(l.valor)})" for l in fora.itertuples())
print(f"Outliers: {meses_fora} acima do limite superior ({diagnostico['limite_superior']}). São meses atípicos, não prova de erro.")

In [ ]:
# ETAPA 12 — SSP/SC (fonte SEPARADA): leitura e conferência da violência doméstica
# Três relatórios oficiais (2023 e 2024). O módulo ssp_sc lê os PDFs, pareia as páginas e confere cada célula contra a tabela do PDF.
# Não foi identificada uma definição operacional de "Fato Comunicado" nos documentos analisados (isso não significa que a definição não exista oficialmente).
# Por isso a unidade não é lida como vítima nem como ocorrência única. Fonte em validação, fora do painel.
from src.pipeline import ssp_sc

resultado_ssp = None
if ssp_sc.acervo_disponivel():
    resultado_ssp = ssp_sc.processar()   # roda em memória: não grava nem altera nenhum dado
    vd = resultado_ssp["vd_extraida"]
    # Conferências: 216 células = 9 categorias × 24 meses, sem chave repetida e com 27 zeros publicados.
    assert len(vd) == 216 and not vd.duplicated(["ano", "mes", "categoria"]).any()
    assert vd["fatos_comunicados"].eq(0).sum() == 27
    assert resultado_ssp["verificacoes"]["ok"].all()
    print("SSP/SC: 216 células; 9 categorias; 24 meses; 27 zeros; unidade em validação.")
    print("Conferências aprovadas:", resultado_ssp["resumo"]["verificacoes_aprovadas"])
    display(resultado_ssp["rastreabilidade"].head())
else:
    print("SSP/SC NÃO EXECUTADA: requer os 45 PDFs e 45 CSVs do acervo local. Nenhum valor foi estimado.")

In [ ]:
# ETAPA 13 — SSP/SC: estatísticas por categoria
# Somamos os 12 meses de cada categoria e ano. As categorias NÃO são somadas entre si.
if resultado_ssp is not None:
    grupos_vd = vd.groupby(["categoria", "ano"])["fatos_comunicados"]
    assert grupos_vd.count().eq(12).all()          # cada categoria/ano precisa ter exatamente 12 meses
    totais_vd = grupos_vd.sum(min_count=12).unstack("ano")
    totais_vd["diferença 2024 − 2023"] = totais_vd[2024] - totais_vd[2023]
    display(totais_vd.sort_values(2024, ascending=False))

    # Média, mediana, mínimo e máximo dos 24 meses; desvio-padrão com ddof=0 (descreve os meses observados).
    por_categoria = vd.groupby("categoria")["fatos_comunicados"]
    estatisticas_vd = por_categoria.agg(meses="count", media="mean", mediana="median", minimo="min", maximo="max")
    estatisticas_vd["desvio_padrao"] = por_categoria.std(ddof=0)
    display(estatisticas_vd.round(2))

In [ ]:
# ETAPA 14 — SSP/SC: gráfico de violência doméstica 2023 × 2024 e leitura
# Cada categoria é comparada apenas consigo mesma. Também fica salvo em documentacao/graficos/analises/07-... (não integra o painel).
if resultado_ssp is not None:
    display(ssp_sc.figura_violencia_domestica(vd))

    mais_em_2024 = int((totais_vd["diferença 2024 − 2023"] > 0).sum())
    menos_em_2024 = int((totais_vd["diferença 2024 − 2023"] < 0).sum())
    print(f"Ameaça: {int(totais_vd.loc['Ameaça', 2023])} em 2023 e {int(totais_vd.loc['Ameaça', 2024])} em 2024 fatos comunicados.")
    print(f"{mais_em_2024} categorias têm mais registros em 2024 e {menos_em_2024} têm menos.")
    print("Isso descreve registros publicados; não demonstra variação do número de vítimas nem explica causas.")

In [ ]:
# ETAPA 15 — SSP/SC: rastreabilidade e republicações dos boletins de homicídio
# Os boletins publicam 294 valores em 58 grupos de republicação. Nenhuma versão é escolhida.
if resultado_ssp is not None:
    display(resultado_ssp["grupos"].groupby("classificacao").size().to_frame("grupos"))
    # Nos 30 valores publicados uma única vez, os campos de grupo ficam ausentes: nunca viram zero nem "não divergente".
    sem_grupo = resultado_ssp["versoes"]["grupo_republicacao"].isna()
    assert sem_grupo.sum() == 30
    assert resultado_ssp["versoes"].loc[sem_grupo, "grupo_divergente"].isna().all()
    # Nenhuma união definitiva Sinesp + SSP/SC foi gravada.
    assert not (RAIZ / "dados/processados/joinville_indicadores_consolidados.csv").exists()
    print("Conferido: 30 valores sem grupo continuam ausentes e não existe CSV consolidado das duas fontes.")

In [ ]:
# ETAPA 16 — Conclusões e limitações
print("CONCLUSÕES")
print(f"- Sinesp: homicídio doloso passou de {int(total_por_ano[2020])} vítimas (2020) para {int(total_por_ano[2025])} (2025) nos anos completos;")
print(f"  em janeiro–setembro, {int(total_periodo[2025])} (2025) e {int(total_periodo[2026])} (2026).")
print("- As séries são contagens mensais pequenas: a diferença absoluta é mais informativa que o percentual.")
print(f"- O critério IQR só é informativo em {informativas} das {len(diagnosticos)} séries.")
if resultado_ssp is not None:
    print("- SSP/SC: exploração separada de violência doméstica (fatos comunicados), extração rastreável e unidade em validação.")
else:
    print("- SSP/SC: não executada neste ambiente (acervo ausente).")

print("\nLIMITAÇÕES")
print("- Registros administrativos agregados: não medem toda a criminalidade, não estimam subnotificação e não indicam causa.")
print("- Sem taxas populacionais, bairro, horário ou corporação; sem total geral entre indicadores ou abrangências.")
print("- 2026 é parcial e a PRF termina antes das demais séries.")
print("- Furto, roubo, estupro e tráfico não existem no Sinesp de Joinville; publicações municipais divergem e não entram na base.")
print("- SSP/SC: não foi identificada uma definição operacional de 'Fato Comunicado' nos documentos analisados (isso não significa que a definição não exista oficialmente); versões de homicídio divergentes; fora do painel; sem soma com o Sinesp.")
print("- Próximos passos: esclarecer 'Fato Comunicado', resolver versões de homicídio e confirmar condições de reutilização.")